In [0]:
%sql
insert into pricing_analytics.gold.pricing_fact_data
select
silverDim.ROW_ID,
dateDim.DATE_ID,
stateDim.STATE_ID,
marketDim.MARKET_ID,
productDim.PRODUCT_ID,
silverDim.ARRIVAL_IN_TONNES,
silverDim.MINIMUM_PRICE,
silverDim.MAXIMUM_PRICE,
silverDim.MODAL_PRICE,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.daily_pricing silverDim
left join pricing_analytics.gold.date_dim dateDim
on silverDim.DATE_OF_PRICING=dateDim.CALENDER_DATE
left join pricing_analytics.gold.state_dim stateDim
on silverDim.STATE_NAME=stateDim.STATE_NAME
left join pricing_analytics.gold.market_dim marketDim
on silverDim.MARKET_NAME=marketDim.MARKET_NAME
left join pricing_analytics.gold.product_dim productDim
on silverDim.PRODUCT_NAME=productDim.PRODUCT_NAME
where silverDim.LAKE_HOUSE_UPDATED_DATE>(select coalesce(max(process_table_date_time), date('1900-01-01')) from pricing_analytics.bronze.pipeline_control_logs where process_name='pricing_data_fact_data' and process_status='Completed')

In [0]:
%sql
insert into pricing_analytics.bronze.pipeline_control_logs(
  select
  'pricing_data_fact_data',
  max(LAKE_HOUSE_UPDATED_DATE),
  'Completed',
  current_timestamp()
  from pricing_analytics.silver.daily_pricing
)